## Step 1 — Install Dependencies

Run the cell below to install all required packages. This only needs to be done **once**.

Key packages:
- `yfinance` — free stock data API
- `xgboost`, `lightgbm` — gradient boosting models
- `torch` — PyTorch for LSTM / Transformer
- `shap` — model explainability
- `plotly` — interactive charts

## Step 1 — Install Dependencies

Run the cell below to install all required packages. This only needs to be done **once**.

Key packages:
- `yfinance` — free stock data API
- `pandas_ta` — 130+ technical indicators
- `xgboost`, `lightgbm` — gradient boosting models
- `torch` — PyTorch for LSTM / Transformer
- `shap` — model explainability
- `plotly` — interactive charts

In [1]:
# Run this ONCE to install all packages
# After running, RESTART the kernel before running subsequent cells

import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ All packages installed.")
    print("\n⚠️  IMPORTANT: Restart the kernel now before running the next cells!")
    print("   (Kernel → Restart Kernel, or click the 🔄 button)")
else:
    print(f"❌ Installation failed:\n{result.stderr}")

✅ All packages installed.

⚠️  IMPORTANT: Restart the kernel now before running the next cells!
   (Kernel → Restart Kernel, or click the 🔄 button)


## Step 2 — Verify Imports

Let's make sure all critical libraries import without errors.

In [2]:
import os, sys, gc, time

# Ensure the project root is on the Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Core
import numpy as np
import pandas as pd
import yaml

# Data
import yfinance as yf

# Models
import xgboost as xgb
import lightgbm as lgb
import torch

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Explainability
import shap

# Sklearn
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, accuracy_score

# Stabilize kernel after heavy imports
gc.collect()
time.sleep(0.2)

print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"XGBoost    : {xgb.__version__}")
print(f"LightGBM   : {lgb.__version__}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
print(f"\n✅ All imports successful.")

NumPy      : 2.4.2
Pandas     : 2.3.3
XGBoost    : 3.1.3
LightGBM   : 4.6.0
PyTorch    : 2.10.0
CUDA avail : False

✅ All imports successful.


## Step 3 — Verify Project Modules

Now let's check that our custom `src/` modules load correctly.
If any import fails, the project structure may be incomplete — re-check the README.

In [3]:
# Data modules
from src.data import TickerUniverse, DataDownloader, DataCleaner

# Feature modules
from src.features import FeaturePipeline

# Backtest modules
from src.backtest import WalkForwardCV, GrowwCostCalculator, BacktestEngine
from src.backtest.metrics import compute_all_metrics

# Model modules
from src.models import XGBoostModel, LightGBMModel, LSTMModel, TransformerModel, EnsembleModel

# Portfolio modules
from src.portfolio import DailyAllocator, PositionSizer, RiskManager

# Options modules
from src.options import OptionsSignalGenerator, BlackScholesGreeks

# Utils
from src.utils.constants import *
from src.utils.helpers import print_disclaimer

print("✅ All project modules imported successfully!")
print(f"\nProject root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Config dir:   {CONFIG_DIR}")

✅ All project modules imported successfully!

Project root: /Users/anto/Trading_Project/masters_trading_ai
Data dir:     /Users/anto/Trading_Project/masters_trading_ai/data
Config dir:   /Users/anto/Trading_Project/masters_trading_ai/config


## Step 4 — Load & Inspect Configuration

All project parameters live in `config/*.yaml`. Let's load them and review key settings.

In [4]:
# Load all config files
with open(CONFIG_DIR / "settings.yaml") as f:
    settings = yaml.safe_load(f)

with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers_cfg = yaml.safe_load(f)

with open(CONFIG_DIR / "groww_fees.yaml") as f:
    fees_cfg = yaml.safe_load(f)

with open(CONFIG_DIR / "model_params.yaml") as f:
    model_cfg = yaml.safe_load(f)

# ---- Key settings ----
print("=" * 50)
print("KEY SETTINGS")
print("=" * 50)
print(f"Risk-free rate:      {settings['market']['risk_free_rate']:.1%}")
print(f"Trading days/year:   {settings['market']['trading_days_per_year']}")
print(f"Prediction horizons: {settings['horizons']} days")
print(f"WF-CV train window:  {settings['walk_forward']['train_window_days']} days")
print(f"WF-CV test window:   {settings['walk_forward']['test_window_days']} days")
print(f"WF-CV embargo:       {settings['walk_forward']['embargo_days']} days")
print(f"WF-CV folds:         {settings['walk_forward']['n_folds']}")
print(f"Max daily drawdown:  {settings['risk']['max_daily_drawdown']:.0%}")
print(f"Max total drawdown:  {settings['risk']['max_total_drawdown']:.0%}")
print(f"Max position size:   {settings['backtest']['max_position_pct']:.0%}")

KEY SETTINGS
Risk-free rate:      6.5%
Trading days/year:   252
Prediction horizons: [1, 5, 10, 20] days
WF-CV train window:  504 days
WF-CV test window:   63 days
WF-CV embargo:       5 days
WF-CV folds:         8
Max daily drawdown:  3%
Max total drawdown:  15%
Max position size:   12%


## Step 5 — Inspect Ticker Universe

We trade **120+ NSE stocks** grouped into 5 buckets:
- **Large Cap** — Blue chips (Reliance, TCS, HDFC Bank...)
- **Banking** — All major banks & NBFCs
- **Mid Cap** — Growing companies with higher volatility
- **High Volatility** — Momentum / speculative names
- **Commodities & Energy** — Mining, oil, gas, metals

In [5]:
universe = TickerUniverse()
print(universe.summary())
print(f"\nTotal unique tickers: {len(universe.get_tickers())}")
print(f"\nFirst 10 tickers: {universe.get_tickers()[:10]}")

            Ticker       Bucket
0      RELIANCE.NS    large_cap
1           TCS.NS    large_cap
2      HDFCBANK.NS    large_cap
3          INFY.NS    large_cap
4     ICICIBANK.NS    large_cap
..             ...          ...
119    BOSCHLTD.NS  commodities
120    DIVISLAB.NS  commodities
121   ZYDUSLIFE.NS  commodities
122  BAJAJ-AUTO.NS  commodities
123        SAIL.NS  commodities

[124 rows x 2 columns]

Total unique tickers: 124

First 10 tickers: ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS', 'HINDUNILVR.NS', 'ITC.NS', 'BHARTIARTL.NS', 'SBIN.NS', 'BAJFINANCE.NS']


## Step 6 — Inspect Groww Fee Structure

Groww charges different fees for equity delivery, intraday, futures, and options.
Our backtester includes **all** of these costs — this is critical for realistic PnL.

In [6]:
cost_calc = GrowwCostCalculator()
print(cost_calc.summary_table())

# Example: Buy ₹50,000 of RELIANCE, sell at ₹52,000
buy_cost = cost_calc.buy_cost(50_000, trade_type="equity_delivery")
sell_cost = cost_calc.sell_cost(52_000, trade_type="equity_delivery")
total_cost = buy_cost.total + sell_cost.total
print(f"\n--- Example: Buy ₹50K → Sell ₹52K (Delivery) ---")
print(f"Buy-side cost:  ₹{buy_cost.total:,.2f}")
print(f"Sell-side cost: ₹{sell_cost.total:,.2f}")
print(f"Total cost:     ₹{total_cost:,.2f}")
print(f"Cost as % of turnover: {total_cost / (50_000 + 52_000) * 100:.3f}%")

{'equity_delivery': {'Total Cost (₹)': '₹174.23', 'Cost %': '0.348%', 'Brokerage': '₹40.00', 'STT': '₹100.00', 'GST': '₹7.73', 'Stamp Duty': '₹7.50', 'DP Charges': '₹15.93'}, 'equity_intraday': {'Total Cost (₹)': '₹64.80', 'Cost %': '0.130%', 'Brokerage': '₹40.00', 'STT': '₹12.50', 'GST': '₹7.73', 'Stamp Duty': '₹1.50', 'DP Charges': '₹0.00'}, 'futures': {'Total Cost (₹)': '₹60.34', 'Cost %': '0.121%', 'Brokerage': '₹40.00', 'STT': '₹10.00', 'GST': '₹7.51', 'Stamp Duty': '₹1.00', 'DP Charges': '₹0.00'}, 'options': {'Total Cost (₹)': '₹157.21', 'Cost %': '0.314%', 'Brokerage': '₹40.00', 'STT': '₹50.00', 'GST': '₹16.11', 'Stamp Duty': '₹1.50', 'DP Charges': '₹0.00'}}

--- Example: Buy ₹50K → Sell ₹52K (Delivery) ---
Buy-side cost:  ₹82.90
Sell-side cost: ₹93.40
Total cost:     ₹176.31
Cost as % of turnover: 0.173%


## Step 7 — Verify Directory Structure

Make sure all data directories exist before we start downloading.

In [7]:
from pathlib import Path

dirs_to_check = [
    DATA_DIR / "raw",
    DATA_DIR / "interim",
    DATA_DIR / "processed",
    DATA_DIR / "external",
    MODELS_DIR,
    REPORTS_DIR,
]

all_ok = True
for d in dirs_to_check:
    exists = d.exists()
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {d}")
    if not exists:
        d.mkdir(parents=True, exist_ok=True)
        print(f"       → Created!")
        all_ok = True

print(f"\n{'✅ All directories ready!' if all_ok else '⚠️ Some directories were missing (now created).'}")

  ✅  /Users/anto/Trading_Project/masters_trading_ai/data/raw
  ✅  /Users/anto/Trading_Project/masters_trading_ai/data/interim
  ✅  /Users/anto/Trading_Project/masters_trading_ai/data/processed
  ✅  /Users/anto/Trading_Project/masters_trading_ai/data/external
  ✅  /Users/anto/Trading_Project/masters_trading_ai/models
  ❌ MISSING  /Users/anto/Trading_Project/masters_trading_ai/reports
       → Created!

✅ All directories ready!


## Step 8 — Walk-Forward CV Preview

This is the heart of our methodology. Walk-Forward Cross-Validation ensures:
- **No look-ahead bias**: We never train on future data
- **Embargo period**: 5-day gap between train and test to prevent label leakage
- **Purge period**: Removes overlapping samples near boundaries

```
 Fold 1: [=== TRAIN (504d) ===]---embargo---[TEST (63d)]
 Fold 2:     [=== TRAIN (504d) ===]---embargo---[TEST (63d)]
 Fold 3:         [=== TRAIN (504d) ===]---embargo---[TEST (63d)]
 ...sliding forward in time...
```

In [9]:
# Create a dummy DataFrame to visualize the splits
dates = pd.bdate_range("2019-01-01", periods=1260, freq="B")  # ~5 years
dummy_df = pd.DataFrame({"x": range(len(dates))}, index=dates)
wfcv = WalkForwardCV()

# Show fold boundaries
splits = list(wfcv.split(dummy_df))
print(f"Number of folds: {len(splits)}")
for i, (train_idx, test_idx) in enumerate(splits):
    if len(train_idx) == 0 or len(test_idx) == 0:
        print(f"  Fold {i+1}: SKIPPED (empty split)")
        continue
    print(f"  Fold {i+1}: Train {dates[train_idx[0]].date()} → {dates[train_idx[-1]].date()} "
          f"({len(train_idx)} days)  |  Test {dates[test_idx[0]].date()} → {dates[test_idx[-1]].date()} "
          f"({len(test_idx)} days)")

Number of folds: 18
  Fold 1: Train 2019-01-01 → 2020-11-27 (499 days)  |  Test 2020-12-14 → 2021-03-10 (63 days)
  Fold 2: Train 2019-01-01 → 2021-02-24 (562 days)  |  Test 2021-03-11 → 2021-06-07 (63 days)
  Fold 3: Train 2019-01-01 → 2021-05-24 (625 days)  |  Test 2021-06-08 → 2021-09-02 (63 days)
  Fold 4: Train 2019-01-01 → 2021-08-19 (688 days)  |  Test 2021-09-03 → 2021-11-30 (63 days)
  Fold 5: Train 2019-01-01 → 2021-11-16 (751 days)  |  Test 2021-12-01 → 2022-02-25 (63 days)
  Fold 6: Train 2019-01-01 → 2022-02-11 (814 days)  |  Test 2022-02-28 → 2022-05-25 (63 days)
  Fold 7: Train 2019-01-01 → 2022-05-11 (877 days)  |  Test 2022-05-26 → 2022-08-22 (63 days)
  Fold 8: Train 2019-01-01 → 2022-08-08 (940 days)  |  Test 2022-08-23 → 2022-11-17 (63 days)
  Fold 9: Train 2019-01-01 → 2022-11-03 (1003 days)  |  Test 2022-11-18 → 2023-02-14 (63 days)
  Fold 10: Train 2019-01-01 → 2023-01-31 (1066 days)  |  Test 2023-02-15 → 2023-05-12 (63 days)
  Fold 11: Train 2019-01-01 → 2023-04

## ✅ Environment Ready!

Everything is set up. Proceed to **Notebook 01 — Data Download** to fetch historical data.

---

*This project is for educational purposes only. Not financial advice.*